# Milestone 2: Conditional generation, guidance & noise-schedule tuning
**From Words to Worlds: Text-to-Image Generation** · IE7615

| Task | Section |
|---|---|
| 1. Integrate text encoder (CLIP / BERT) with the diffusion model | §1 |
| 2. Run baseline conditional generation | §2 |
| 3. Tune classifier-free guidance + noise schedule | §3 |
| 4. Record first set of generated images | §4 |

Reusable code (diffusion loop, schedulers, CLIPScore, logging, grids) is defined once in §0.1.
Everything is logged to `results/logs/generation_log.csv`; images go to `results/samples/m2_tuning/`, metrics to `results/metrics/`, plots to `results/plots/`.

Runtime on a Colab T4: about 25–30 min end to end (most of it is the scheduler sweep in §3.2).

## 0. Setup
Same as milestone 1: mount Drive (the cleaned COCO data lives there), clone/pull the repo, install pinned requirements.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Aryan16d/IE7615-Genrative-milestone.git"
REPO_DIR = "/content/IE7615-Genrative-milestone"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%pip install -q -r {REPO_DIR}/requirements.txt

## 0.1 Helper functions
Diffusion loop, schedulers, CLIPScore, logging and grids, used by every section below.

In [ ]:
import inspect
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import textwrap
from PIL import Image
from diffusers import (
    PNDMScheduler, DDIMScheduler, DDPMScheduler, EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler, DPMSolverMultistepScheduler,
)

# ---------------------------------------------------------------- prompts
def load_prompts(path):
    return [l.strip() for l in open(path) if l.strip()]


# ---------------------------------------------------------------- schedulers (= how we walk the noise schedule)
SCHEDULERS = {
    "PNDM (SD default)": lambda c: PNDMScheduler.from_config(c),
    "DDIM":              lambda c: DDIMScheduler.from_config(c),
    "DDPM":              lambda c: DDPMScheduler.from_config(c),
    "Euler":             lambda c: EulerDiscreteScheduler.from_config(c),
    "Euler (Karras)":    lambda c: EulerDiscreteScheduler.from_config(c, use_karras_sigmas=True),
    "Euler a":           lambda c: EulerAncestralDiscreteScheduler.from_config(c),
    "DPM++ 2M":          lambda c: DPMSolverMultistepScheduler.from_config(c, algorithm_type="dpmsolver++"),
    "DPM++ 2M Karras":   lambda c: DPMSolverMultistepScheduler.from_config(c, algorithm_type="dpmsolver++",
                                                                          use_karras_sigmas=True),
}

def make_scheduler(name, base_config):
    """Fresh scheduler every call: PNDM / DPM++ keep internal state between steps."""
    return SCHEDULERS[name](base_config)


# ---------------------------------------------------------------- text encoder -> conditioning
@torch.no_grad()
def encode_text(tokenizer, text_encoder, prompts, device):
    """prompts -> [B, 77, hidden] per-token embeddings (what the UNet cross-attends to)."""
    ids = tokenizer(prompts, padding="max_length", max_length=77,
                    truncation=True, return_tensors="pt").input_ids.to(device)
    return text_encoder(input_ids=ids).last_hidden_state


# ---------------------------------------------------------------- diffusion loop (UNet + scheduler + CFG)
def _initial_latents(seeds, shape, device, dtype):
    lat = [torch.randn((1, *shape), generator=torch.Generator(device).manual_seed(int(s)),
                       device=device, dtype=dtype) for s in seeds]
    return torch.cat(lat)

@torch.no_grad()
def denoise(unet, scheduler, cond, uncond, guidance, steps, seeds, height=512, width=512):
    """Classifier-free guided reverse diffusion in latent space.
    eps = eps_uncond + w * (eps_cond - eps_uncond); w <= 1 means no guidance (cond only)."""
    device, dtype = cond.device, cond.dtype
    scheduler.set_timesteps(steps, device=device)
    latents = _initial_latents(seeds, (unet.config.in_channels, height // 8, width // 8), device, dtype)
    latents = latents * scheduler.init_noise_sigma

    do_cfg = guidance > 1.0
    emb = torch.cat([uncond, cond]) if do_cfg else cond
    step_kwargs = {}
    if "generator" in inspect.signature(scheduler.step).parameters:   # ancestral samplers add noise
        step_kwargs["generator"] = torch.Generator(device).manual_seed(int(seeds[0]))

    for t in scheduler.timesteps:
        x = torch.cat([latents] * 2) if do_cfg else latents
        x = scheduler.scale_model_input(x, t)
        eps = unet(x, t, encoder_hidden_states=emb).sample
        if do_cfg:
            eps_u, eps_c = eps.chunk(2)
            eps = eps_u + guidance * (eps_c - eps_u)
        latents = scheduler.step(eps, t, latents, **step_kwargs).prev_sample
    return latents

@torch.no_grad()
def decode(vae, latents):
    img = vae.decode(latents / vae.config.scaling_factor).sample
    img = (img / 2 + 0.5).clamp(0, 1).permute(0, 2, 3, 1).float().cpu().numpy()
    return [Image.fromarray((x * 255).round().astype("uint8")) for x in img]

def nsfw_flags(pipe, images):
    """Run SD's safety checker on our own outputs (we bypass pipe(), so it isn't applied automatically)."""
    if getattr(pipe, "safety_checker", None) is None:
        return [False] * len(images)
    try:
        inp = pipe.feature_extractor(images, return_tensors="pt").pixel_values.to(pipe.device, pipe.dtype)
        arr = np.stack([np.asarray(i, dtype=np.float32) / 255 for i in images])
        _, flags = pipe.safety_checker(images=arr, clip_input=inp)
        return [bool(f) for f in flags]
    except Exception as e:
        print("safety check skipped:", e)
        return [False] * len(images)

def generate(pipe, cond, uncond, seeds, scheduler_name="DPM++ 2M Karras", steps=25, guidance=7.5,
             height=512, width=512):
    """Full text-conditioned generation from precomputed embeddings.
    Returns (images, seconds per image, nsfw flags). Flagged images are replaced with black."""
    sched = make_scheduler(scheduler_name, pipe.scheduler.config)
    uncond = uncond.expand(len(cond), -1, -1)
    if cond.device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    lat = denoise(pipe.unet, sched, cond, uncond, guidance, steps, seeds, height, width)
    imgs = decode(pipe.vae, lat)
    if cond.device.type == "cuda": torch.cuda.synchronize()
    dt = (time.time() - t0) / len(cond)
    flags = nsfw_flags(pipe, imgs)
    imgs = [Image.new("RGB", im.size) if f else im for im, f in zip(imgs, flags)]
    return imgs, dt, flags


# ---------------------------------------------------------------- metrics
class ClipScorer:
    """CLIPScore = 100 * cos(CLIP image emb, CLIP text emb), clipped at 0 (higher = closer to prompt)."""
    def __init__(self, model_id, device, dtype):
        from transformers import CLIPModel, CLIPProcessor
        self.m = CLIPModel.from_pretrained(model_id, dtype=dtype).to(device).eval()
        self.p = CLIPProcessor.from_pretrained(model_id)
        self.device, self.dtype = device, dtype

    @torch.no_grad()
    def image_emb(self, images):
        images = [Image.open(i).convert("RGB") if isinstance(i, (str, Path)) else i for i in images]
        px = self.p(images=images, return_tensors="pt")["pixel_values"].to(self.device, self.dtype)
        e = self.m.visual_projection(self.m.vision_model(pixel_values=px).pooler_output)
        return (e / e.norm(dim=-1, keepdim=True)).float().cpu()

    @torch.no_grad()
    def text_emb(self, texts):
        t = self.p(text=list(texts), padding=True, truncation=True, max_length=77, return_tensors="pt").to(self.device)
        e = self.m.text_projection(self.m.text_model(input_ids=t["input_ids"],
                                                     attention_mask=t["attention_mask"]).pooler_output)
        return (e / e.norm(dim=-1, keepdim=True)).float().cpu()

    def score(self, images, texts):
        s = (self.image_emb(images) * self.text_emb(texts)).sum(-1).clamp(min=0) * 100
        return [round(float(x), 2) for x in s]

def mean_saturation(img):
    """0-1 HSV saturation; rises sharply when guidance is too high (over-saturated, 'burnt' images)."""
    return round(float(np.asarray(img.convert("HSV"))[..., 1].mean() / 255), 4)


# ---------------------------------------------------------------- logging (results/logs/generation_log.csv)
LOG_COLUMNS = ["run_id", "timestamp", "milestone", "prompt", "seed", "model_id", "scheduler", "num_steps",
               "guidance_scale", "height", "width", "output_path", "runtime_s", "notes"]

class GenLog:
    def __init__(self, csv_path, repo_dir, milestone, model_id):
        self.csv, self.repo, self.ms, self.model = Path(csv_path), Path(repo_dir), milestone, model_id
        self.stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        self.rows, self.n = [], 0

    def add(self, prompt, seed, scheduler, steps, guidance, h, w, path, runtime_s, notes=""):
        self.n += 1
        try: rel = str(Path(path).relative_to(self.repo))
        except ValueError: rel = str(path)
        self.rows.append(dict(run_id=f"{self.ms.lower()}-{self.stamp}-{self.n:03d}",
                              timestamp=datetime.now().isoformat(timespec="seconds"), milestone=self.ms,
                              prompt=prompt, seed=seed, model_id=self.model, scheduler=scheduler,
                              num_steps=steps, guidance_scale=guidance, height=h, width=w,
                              output_path=rel, runtime_s=round(runtime_s, 2), notes=notes))

    def flush(self):
        if not self.rows: return 0
        self.csv.parent.mkdir(parents=True, exist_ok=True)
        header = not self.csv.exists() or self.csv.stat().st_size == 0
        pd.DataFrame(self.rows, columns=LOG_COLUMNS).to_csv(self.csv, mode="a", index=False, header=header)
        n, self.rows = len(self.rows), []
        return n


# ---------------------------------------------------------------- figures
def save_grid(images, path, row_labels=None, col_labels=None, cell=3.0, dpi=70, title=None, wrap=28):
    """images: list of rows (list of PIL). Saves and returns path."""
    R, C = len(images), max(len(r) for r in images)
    fig, ax = plt.subplots(R, C, figsize=(cell * C, cell * R + (0.4 if title else 0)), squeeze=False)
    for i in range(R):
        for j in range(C):
            a = ax[i, j]; a.set_xticks([]); a.set_yticks([])
            for s in a.spines.values(): s.set_visible(False)
            if j < len(images[i]) and images[i][j] is not None:
                a.imshow(images[i][j])
            if i == 0 and col_labels: a.set_title(textwrap.fill(str(col_labels[j]), wrap), fontsize=9)
            if j == 0 and row_labels: a.set_ylabel(textwrap.fill(str(row_labels[i]), wrap), fontsize=8)
    if title: fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=dpi); plt.close(fig)
    return path

In [ ]:
import json, time, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import Image as IPImage, display

SEED = 39
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
H = W = 512

DRIVE_ROOT  = Path("/content/drive/MyDrive/t2i")
PROC_DIR    = DRIVE_ROOT / "data" / "processed"
IMG_OUT     = PROC_DIR / "images"
REPO_DIR    = Path(REPO_DIR)
RESULTS_DIR = REPO_DIR / "results"
SAMPLES_DIR = RESULTS_DIR / "samples" / "m2_tuning"
METRICS_DIR = RESULTS_DIR / "metrics"
PLOTS_DIR   = RESULTS_DIR / "plots"
LOG_CSV     = RESULTS_DIR / "logs" / "generation_log.csv"
for d in (SAMPLES_DIR, METRICS_DIR, PLOTS_DIR): d.mkdir(parents=True, exist_ok=True)

SD_ID   = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CLIP_ID = "openai/clip-vit-large-patch14"

prompts = load_prompts(REPO_DIR / "prompts" / "test_prompts.txt")
print(f"{len(prompts)} test prompts | device {DEVICE} | torch {torch.__version__}")
for i, p in enumerate(prompts, 1): print(f"  {i}. {p}")

## 1. Integrating the text encoder with the diffusion model

Stable Diffusion is a **latent** diffusion model with four parts. In M1 we used `pipe(prompt)`, which hides how they connect. Here we wire them up ourselves:

```
prompt ──tokenizer──► 77 token ids ──CLIP text encoder──► c ∈ R^{77×768}
                                                             │ (cross-attention keys/values)
z_T ~ N(0, I) ∈ R^{4×64×64} ──► UNet ε_θ(z_t, t, c) ──scheduler step──► z_{t-1} … ► z_0 ──VAE decoder──► 512×512 image
```

**Classifier-free guidance (CFG).** Each step runs the UNet twice, once with the prompt embedding `c` and once with the empty-prompt embedding `∅`, and extrapolates:

  ε̂ = ε_θ(z_t, t, ∅) + w · (ε_θ(z_t, t, c) − ε_θ(z_t, t, ∅))

`w = 1` is plain conditional sampling; larger `w` pushes harder toward the prompt. The **scheduler** decides which timesteps / noise levels we visit and how each step updates `z_t` (§3.2).

Our own loop is `denoise()` + `decode()` from §0.1.

In [ ]:
from diffusers import StableDiffusionPipeline
pipe = StableDiffusionPipeline.from_pretrained(SD_ID, dtype=DTYPE, variant="fp16").to(DEVICE)
pipe.set_progress_bar_config(disable=True)
tokenizer, text_encoder, unet, vae = pipe.tokenizer, pipe.text_encoder, pipe.unet, pipe.vae

n_params = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"text encoder : {type(text_encoder).__name__:22s} {n_params(text_encoder):7.1f} M params | hidden {text_encoder.config.hidden_size}")
print(f"UNet         : {type(unet).__name__:22s} {n_params(unet):7.1f} M params | cross_attention_dim {unet.config.cross_attention_dim}")
print(f"VAE          : {type(vae).__name__:22s} {n_params(vae):7.1f} M params | latent channels {vae.config.latent_channels}, scale {vae.config.scaling_factor}")
print(f"scheduler    : {type(pipe.scheduler).__name__} | {pipe.scheduler.config.num_train_timesteps} train steps, beta schedule '{pipe.scheduler.config.beta_schedule}'")
assert unet.config.cross_attention_dim == text_encoder.config.hidden_size, "text encoder width must match UNet cross-attention"

In [ ]:
# text -> conditioning: one 77x768 matrix per prompt, plus the empty prompt for CFG
cond   = encode_text(tokenizer, text_encoder, prompts, DEVICE)   # [5, 77, 768]
uncond = encode_text(tokenizer, text_encoder, [""], DEVICE)      # [1, 77, 768]
print("cond:", tuple(cond.shape), cond.dtype, "| uncond:", tuple(uncond.shape))

# how different is each prompt from the empty prompt? (this gap is what CFG amplifies)
gap = (cond.float() - uncond.float()).norm(dim=-1).mean(-1).cpu()
for p, g in zip(prompts, gap): print(f"  ||c - ∅|| = {g:5.2f}   {p}")

### 1.1 Sanity check: our loop reproduces `pipe()`
Same prompt, seed 39, PNDM, 50 steps, guidance 7.5 (the M1 setting). If the integration is right, the two images should be (nearly) identical; tiny differences come from fp16 GPU kernels.

In [ ]:
ref = pipe(prompts[0], num_inference_steps=50, guidance_scale=7.5, height=H, width=W,
           generator=torch.Generator(DEVICE).manual_seed(SEED)).images[0]
ours, _, _ = generate(pipe, cond[:1], uncond, [SEED], "PNDM (SD default)", steps=50, guidance=7.5, height=H, width=W)
ours = ours[0]

diff = np.abs(np.asarray(ref, dtype=np.int16) - np.asarray(ours, dtype=np.int16))
print(f"pixel diff vs pipe(): mean {diff.mean():.3f}, max {diff.max()} (0–255 scale)")
fig, ax = plt.subplots(1, 3, figsize=(13, 4.5))
for a, im, t in zip(ax, [ref, ours, diff.mean(-1)], ["pipe()", "our loop (generate)", "|difference|"]):
    a.imshow(im, cmap="magma" if t.startswith("|") else None); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

### 1.2 Why CLIP and not BERT?
BERT-base also outputs 768-d token vectors, so it *fits* the UNet's cross-attention shape. But SD's UNet was trained only on CLIP text embeddings, and BERT's embedding space is unrelated. Swapping it in without training gives images that ignore the prompt.
Using BERT properly would need a trained projection or fine-tuning the UNet's cross-attention layers on our COCO pairs (a possible extension). Imagen does this with a frozen T5 encoder, but trains the diffusion model on it from scratch.

Set `RUN_BERT_DEMO = False` to skip (saves ~1 min).

In [ ]:
RUN_BERT_DEMO = True
bert_rows = []
if RUN_BERT_DEMO:
    from transformers import BertTokenizer, BertModel
    btok = BertTokenizer.from_pretrained("bert-base-uncased")
    bert = BertModel.from_pretrained("bert-base-uncased", dtype=DTYPE).to(DEVICE).eval()
    with torch.no_grad():
        enc = lambda txt: bert(**btok(txt, padding="max_length", max_length=77, truncation=True,
                                      return_tensors="pt").to(DEVICE)).last_hidden_state
        b_cond, b_uncond = enc(prompts[:3]), enc([""])
    print("BERT embeddings:", tuple(b_cond.shape), "(same shape as CLIP, different space)")
    clip_imgs, _, _ = generate(pipe, cond[:3], uncond, [SEED] * 3, "DPM++ 2M Karras", 25, 7.5, H, W)
    bert_imgs, _, _ = generate(pipe, b_cond, b_uncond, [SEED] * 3, "DPM++ 2M Karras", 25, 7.5, H, W)
    del bert; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 2. Baseline conditional generation
**Baseline = Stable Diffusion's defaults**, run through our integrated loop: PNDM scheduler, 50 steps, guidance 7.5, seed 39 (same as M1).
To show the text conditioning is doing the work, we also sample **unconditionally** (empty prompt, no guidance) from the same starting noise.

Metric: **CLIPScore** (100 × cosine similarity between the image and the prompt in CLIP space; higher = more faithful). For reference, real COCO image–caption pairs score around 25–35 with ViT-L/14.

In [ ]:
scorer = ClipScorer(CLIP_ID, DEVICE, DTYPE)
log = GenLog(LOG_CSV, REPO_DIR, "M2", SD_ID)
metrics = []   # one row per generated image, all experiments

def record(images, prompt_list, seeds, out_dir, experiment, scheduler, steps, guidance, runtime_s,
           flags=None, ext="jpg", tag="", notes=""):
    """Save images, score them, add rows to the generation log and the metrics table."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    scores = scorer.score(images, prompt_list)
    flags = flags or [False] * len(images)
    for k, (img, p, s, cs, fl) in enumerate(zip(images, prompt_list, seeds, scores, flags)):
        pi = prompts.index(p) + 1 if p in prompts else k + 1
        path = out_dir / f"{tag}p{pi:02d}_s{s}.{ext}"
        img.save(path, quality=92) if ext == "jpg" else img.save(path)
        note = notes + ("; safety checker flagged (black image)" if fl else "")
        log.add(p, s, scheduler, steps, guidance, H, W, path, runtime_s, note.strip("; "))
        metrics.append(dict(experiment=experiment, prompt_id=pi, prompt=p, seed=s, scheduler=scheduler,
                            steps=steps, guidance=guidance, clip_score=cs, saturation=mean_saturation(img),
                            runtime_s=round(runtime_s, 2), flagged=fl, path=str(path.relative_to(REPO_DIR))))
    log.flush()   # write after every call so a crash mid-sweep keeps what's done
    return scores

if RUN_BERT_DEMO:
    bdir = SAMPLES_DIR / "encoder_clip_vs_bert"
    record(clip_imgs, prompts[:3], [SEED] * 3, bdir, "encoder: CLIP", "DPM++ 2M Karras", 25, 7.5, 0, tag="clip_")
    record(bert_imgs, prompts[:3], [SEED] * 3, bdir, "encoder: BERT (untrained)", "DPM++ 2M Karras", 25, 7.5, 0,
           tag="bert_", notes="BERT-base embeddings fed to SD UNet without training")
    m = pd.DataFrame(metrics)
    cs_c = m[m.experiment == "encoder: CLIP"].clip_score.values
    cs_b = m[m.experiment.str.startswith("encoder: BERT")].clip_score.values
    g = save_grid([clip_imgs, bert_imgs], PLOTS_DIR / "m2_encoder_clip_vs_bert.png",
                  row_labels=[f"CLIP text encoder\n(mean CLIPScore {cs_c.mean():.1f})",
                              f"BERT-base, no training\n(mean CLIPScore {cs_b.mean():.1f})"],
                  col_labels=prompts[:3], title="Same UNet, same noise, different text encoder")
    display(IPImage(filename=str(g), width=900))

In [ ]:
BASE = dict(scheduler="PNDM (SD default)", steps=50, guidance=7.5)
seeds = [SEED] * len(prompts)

base_imgs, base_dt, base_flags = generate(pipe, cond, uncond, seeds, BASE["scheduler"], BASE["steps"], BASE["guidance"], H, W)
base_scores = record(base_imgs, prompts, seeds, SAMPLES_DIR / "baseline", "baseline (conditional)",
                     **BASE, runtime_s=base_dt, flags=base_flags, ext="png", notes="M2 baseline = SD defaults")

# unconditional: empty prompt as the condition, no guidance -> the model's prior with no text
unc_imgs, unc_dt, unc_flags = generate(pipe, uncond.expand(len(prompts), -1, -1), uncond, seeds,
                                       BASE["scheduler"], BASE["steps"], 1.0, H, W)
unc_scores = record(unc_imgs, prompts, seeds, SAMPLES_DIR / "baseline", "unconditional", BASE["scheduler"],
                    BASE["steps"], 1.0, unc_dt, flags=unc_flags, tag="uncond_",
                    notes="empty prompt, no CFG; scored against the test prompt")

print(f"{base_dt:.1f}s / image")
pd.DataFrame({"prompt": prompts, "conditional": base_scores, "unconditional": unc_scores})

In [ ]:
g = save_grid([base_imgs, unc_imgs], PLOTS_DIR / "m2_baseline_cond_vs_uncond.png",
              row_labels=[f"conditional (w=7.5)\nCLIPScore {np.mean(base_scores):.1f}",
                          f"unconditional (no text)\nCLIPScore {np.mean(unc_scores):.1f}"],
              col_labels=prompts, title="Baseline: SD v1.5, PNDM, 50 steps, seed 39")
display(IPImage(filename=str(g), width=1100))
print(f"text conditioning adds {np.mean(base_scores) - np.mean(unc_scores):+.1f} CLIPScore on average")

## 3. Tuning
### 3.1 Classifier-free guidance scale `w`
We sweep `w` with everything else fixed (DPM++ 2M Karras, 25 steps: a fast, stable sampler, so the sweep only measures guidance).
Two seeds per prompt to average out noise.

What to expect: `w = 1` follows the prompt weakly; CLIPScore rises with `w`, then flattens, while images get **over-saturated** and high-contrast at large `w`. We track mean HSV saturation as a simple proxy for that.

In [ ]:
GUIDANCE_GRID = [1.0, 3.0, 5.0, 7.5, 10.0, 15.0]
CFG_SCHED, CFG_STEPS = "DPM++ 2M Karras", 25
CFG_SEEDS = [SEED, SEED + 1]

cfg_imgs = {}   # (guidance, seed) -> list of images (one per prompt)
for s in CFG_SEEDS:
    for w in GUIDANCE_GRID:
        imgs, dt, fl = generate(pipe, cond, uncond, [s] * len(prompts), CFG_SCHED, CFG_STEPS, w, H, W)
        record(imgs, prompts, [s] * len(prompts), SAMPLES_DIR / "cfg_sweep", "cfg sweep",
               CFG_SCHED, CFG_STEPS, w, dt, flags=fl, tag=f"w{w:g}_")
        cfg_imgs[(w, s)] = imgs
        print(f"seed {s}  w={w:<5g} {dt:4.1f}s/img")

cfg = pd.DataFrame(metrics).query("experiment == 'cfg sweep'")
cfg_summary = (cfg.groupby("guidance").agg(clip_mean=("clip_score", "mean"), clip_std=("clip_score", "std"),
                                           saturation=("saturation", "mean"), runtime_s=("runtime_s", "mean"))
               .round(3).reset_index())
cfg_summary.to_csv(METRICS_DIR / "m2_cfg_sweep_summary.csv", index=False)
cfg_summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].errorbar(cfg_summary.guidance, cfg_summary.clip_mean, yerr=cfg_summary.clip_std, marker="o", capsize=4)
ax[0].set(xlabel="guidance scale w", ylabel="CLIPScore", title="Prompt fidelity vs guidance")
for pid, g_ in cfg.groupby("prompt_id"):
    ax[0].plot(*zip(*g_.groupby("guidance").clip_score.mean().items()), alpha=.25, lw=1)
ax[1].plot(cfg_summary.guidance, cfg_summary.saturation, marker="o", color="tab:red")
ax[1].set(xlabel="guidance scale w", ylabel="mean HSV saturation", title="Over-saturation vs guidance")
plt.tight_layout(); plt.savefig(PLOTS_DIR / "m2_cfg_sweep.png", dpi=110); plt.show()

g = save_grid([[cfg_imgs[(w, SEED)][i] for w in GUIDANCE_GRID] for i in range(len(prompts))],
              PLOTS_DIR / "m2_cfg_grid.png", row_labels=prompts, col_labels=[f"w = {w:g}" for w in GUIDANCE_GRID],
              cell=2.6, title=f"Guidance sweep ({CFG_SCHED}, {CFG_STEPS} steps, seed {SEED})")
display(IPImage(filename=str(g), width=1100))

**Choosing `w`.** CLIPScore alone always prefers large `w`, but those images look burnt. Rule: take the **smallest `w` whose mean CLIPScore is within 0.5 points of the best**. That gets nearly all the prompt fidelity with natural colours.

In [ ]:
best_clip = cfg_summary.clip_mean.max()
BEST_W = float(cfg_summary.loc[cfg_summary.clip_mean >= best_clip - 0.5, "guidance"].min())
print(f"best mean CLIPScore {best_clip:.2f} at w={cfg_summary.loc[cfg_summary.clip_mean.idxmax(), 'guidance']:g}"
      f" -> chosen w = {BEST_W:g} (CLIPScore {cfg_summary.set_index('guidance').clip_mean[BEST_W]:.2f})")

### 3.2 Noise schedule
Three knobs control how we traverse the noise schedule at inference:
1. **Sampler / scheduler**: the update rule. DDPM is stochastic and was designed for ~1000 steps; DDIM is its deterministic version; PNDM is SD's default; Euler and DPM++ are ODE solvers that need far fewer steps; "Euler a" adds fresh noise each step (ancestral).
2. **Sigma spacing**: the default spacing vs **Karras et al. (2022)**, which spends more steps at low noise, where fine details form.
3. **Number of steps**.

SD was trained with a fixed `scaled_linear` β schedule (1000 steps). We don't change the trained schedule, only how we sample from it. The plot below shows both.

In [ ]:
from diffusers import DDPMScheduler, EulerDiscreteScheduler
cfgS = pipe.scheduler.config
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

for beta, ls in [("scaled_linear", "-"), ("linear", "--")]:
    s = DDPMScheduler.from_config(cfgS, beta_schedule=beta)
    ac = s.alphas_cumprod.numpy()
    ax[0].plot(np.log(ac / (1 - ac)), ls, label=f"{beta}" + (" (SD training)" if beta == "scaled_linear" else ""))
ax[0].set(xlabel="training timestep t", ylabel="log SNR", title="Training noise schedule (1000 steps)"); ax[0].legend()

for name in ["Euler", "Euler (Karras)"]:
    s = make_scheduler(name, cfgS); s.set_timesteps(25)
    ax[1].plot(s.sigmas.numpy()[:-1], marker=".", label=name)
ax[1].set(xlabel="sampling step", ylabel="σ (noise level)", yscale="log", title="Sampling noise levels, 25 steps"); ax[1].legend()
plt.tight_layout(); plt.savefig(PLOTS_DIR / "m2_noise_schedules.png", dpi=110); plt.show()

In [ ]:
SCHED_NAMES = list(SCHEDULERS)            # 8 samplers, see SCHEDULERS in §0.1
STEPS_GRID  = [10, 20, 30, 50]
sched_imgs = {}
for name in SCHED_NAMES:
    for n in STEPS_GRID:
        imgs, dt, fl = generate(pipe, cond, uncond, seeds, name, n, BEST_W, H, W)
        tag = name.lower().replace(" ", "").replace("+", "p").replace("(", "_").replace(")", "")
        cs = record(imgs, prompts, seeds, SAMPLES_DIR / "scheduler_sweep", "scheduler sweep",
                    name, n, BEST_W, dt, flags=fl, tag=f"{tag}_{n}steps_")
        sched_imgs[(name, n)] = imgs
        print(f"{name:18s} {n:2d} steps  {dt:4.1f}s/img  CLIP {np.mean(cs):.2f}")

sch = pd.DataFrame(metrics).query("experiment == 'scheduler sweep'")
sched_summary = (sch.groupby(["scheduler", "steps"]).agg(clip_mean=("clip_score", "mean"), clip_std=("clip_score", "std"),
                                                         runtime_s=("runtime_s", "mean")).round(3).reset_index())
sched_summary.to_csv(METRICS_DIR / "m2_scheduler_sweep_summary.csv", index=False)
sched_summary.pivot(index="scheduler", columns="steps", values="clip_mean").loc[SCHED_NAMES]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for name in SCHED_NAMES:
    d = sched_summary[sched_summary.scheduler == name]
    ax[0].plot(d.steps, d.clip_mean, marker="o", label=name)
    ax[1].scatter(d.runtime_s, d.clip_mean, s=20 + d.steps * 2, label=name)
ax[0].set(xlabel="sampling steps", ylabel="CLIPScore", title=f"Sampler × steps (w = {BEST_W:g})")
ax[1].set(xlabel="seconds / image (T4)", ylabel="CLIPScore", title="Quality vs speed (marker size = steps)")
ax[0].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.savefig(PLOTS_DIR / "m2_scheduler_sweep.png", dpi=110); plt.show()

for i, p in enumerate(prompts):
    g = save_grid([[sched_imgs[(nm, n)][i] for n in STEPS_GRID] for nm in SCHED_NAMES],
                  PLOTS_DIR / f"m2_scheduler_grid_p{i+1:02d}.png", row_labels=SCHED_NAMES,
                  col_labels=[f"{n} steps" for n in STEPS_GRID], cell=2.4, title=p)
display(IPImage(filename=str(PLOTS_DIR / "m2_scheduler_grid_p01.png"), width=700))

**Choosing the sampler.** Among configs within 0.5 CLIPScore of the best, pick the **fastest** (fewest seconds per image). That favours a solver that converges in few steps over brute force.

In [ ]:
best_sched_clip = sched_summary.clip_mean.max()
near = sched_summary[sched_summary.clip_mean >= best_sched_clip - 0.5].sort_values(["runtime_s", "clip_mean"],
                                                                                    ascending=[True, False])
BEST = dict(scheduler=near.iloc[0].scheduler, steps=int(near.iloc[0].steps), guidance=BEST_W)
print("configs within 0.5 of the best:"); display(near)
print("chosen config:", BEST)

## 4. First set of generated images (tuned config)
1. The 5 test prompts × 4 seeds with the tuned config (PNG, `results/samples/m2_tuning/final_set/`)
2. 10 held-out COCO **test-split** captions, generated and shown next to the real photo
3. Baseline vs tuned summary for the report

In [ ]:
FINAL_SEEDS = [SEED, SEED + 1, SEED + 2, SEED + 3]
final = {}
for s in FINAL_SEEDS:
    imgs, dt, fl = generate(pipe, cond, uncond, [s] * len(prompts), BEST["scheduler"], BEST["steps"], BEST["guidance"], H, W)
    record(imgs, prompts, [s] * len(prompts), SAMPLES_DIR / "final_set", "final set (tuned)",
           **BEST, runtime_s=dt, flags=fl, ext="png", notes="M2 tuned config")
    final[s] = imgs

g = save_grid([[final[s][i] for s in FINAL_SEEDS] for i in range(len(prompts))],
              SAMPLES_DIR / "final_set" / "grid.png", row_labels=prompts,
              col_labels=[f"seed {s}" for s in FINAL_SEEDS], cell=2.8,
              title=f"M2 tuned: {BEST['scheduler']}, {BEST['steps']} steps, w = {BEST['guidance']:g}")
display(IPImage(filename=str(g), width=900))

In [ ]:
pairs = pd.read_csv(PROC_DIR / "pairs_clean.csv")
pairs["image_path"] = pairs.file_name.map(lambda f: str(IMG_OUT / f))
coco = pairs[pairs.split == "test"].sample(10, random_state=SEED).reset_index(drop=True)

c_cond = encode_text(tokenizer, text_encoder, coco.caption.tolist(), DEVICE)
coco_imgs, coco_dt, coco_fl = [], 0, []
for k in range(0, len(coco), 5):            # batches of 5 to stay well inside T4 memory
    im, dt, fl = generate(pipe, c_cond[k:k+5], uncond, [SEED] * len(c_cond[k:k+5]),
                          BEST["scheduler"], BEST["steps"], BEST["guidance"], H, W)
    coco_imgs += im; coco_fl += fl; coco_dt += dt / 2
gen_cs = record(coco_imgs, coco.caption.tolist(), [SEED] * len(coco), SAMPLES_DIR / "coco_test",
                "coco test (tuned)", **BEST, runtime_s=coco_dt, flags=coco_fl, tag="coco_",
                notes="held-out COCO test-split caption")
real_cs = scorer.score(coco.image_path.tolist(), coco.caption.tolist())
coco_res = coco[["image_id", "caption"]].assign(clip_generated=gen_cs, clip_real=real_cs)
coco_res.to_csv(METRICS_DIR / "m2_coco_test_clipscore.csv", index=False)

real = [Image.open(p).convert("RGB") for p in coco.image_path]
g = save_grid([coco_imgs[:5], real[:5], coco_imgs[5:], real[5:]], PLOTS_DIR / "m2_coco_generated_vs_real.png",
              row_labels=["generated", "real COCO", "generated", "real COCO"],
              col_labels=[f"{c}" for c in coco.caption[:5]], cell=2.8,
              title=f"Held-out COCO captions | CLIPScore generated {np.mean(gen_cs):.1f} vs real {np.mean(real_cs):.1f}")
display(IPImage(filename=str(g), width=1000))
coco_res

In [ ]:
m = pd.DataFrame(metrics)
m.to_csv(METRICS_DIR / "m2_all_generations.csv", index=False)

b = m[m.experiment == "baseline (conditional)"]
t = m[(m.experiment == "final set (tuned)") & (m.seed == SEED)]     # same prompts + seed as the baseline
summary = pd.DataFrame({
    "config":       [f"{BASE['scheduler']}, {BASE['steps']} steps, w={BASE['guidance']:g}",
                     f"{BEST['scheduler']}, {BEST['steps']} steps, w={BEST['guidance']:g}"],
    "clip_mean":    [b.clip_score.mean(), t.clip_score.mean()],
    "saturation":   [b.saturation.mean(), t.saturation.mean()],
    "sec_per_img":  [b.runtime_s.mean(), t.runtime_s.mean()],
}, index=["M1/M2 baseline", "M2 tuned"]).round(3)
summary["speedup"] = (summary.sec_per_img.iloc[0] / summary.sec_per_img).round(2)
display(summary)

g = save_grid([base_imgs, final[SEED]], PLOTS_DIR / "m2_baseline_vs_tuned.png",
              row_labels=[f"baseline\nCLIP {b.clip_score.mean():.1f} · {b.runtime_s.mean():.1f}s",
                          f"tuned\nCLIP {t.clip_score.mean():.1f} · {t.runtime_s.mean():.1f}s"],
              col_labels=prompts, title="Baseline vs tuned (seed 39)")
display(IPImage(filename=str(g), width=1100))

report = {
    "baseline": BASE, "tuned": BEST,
    "cfg_sweep": cfg_summary.to_dict("records"),
    "baseline_vs_tuned": summary.reset_index().to_dict("records"),
    "unconditional_clip_mean": float(np.mean(unc_scores)),
    "coco_test_clip": {"generated": float(np.mean(gen_cs)), "real": float(np.mean(real_cs))},
    "n_images_generated": int(len(m)),
}
(METRICS_DIR / "m2_summary.json").write_text(json.dumps(report, indent=2, default=str))
log.flush()
print(f"{log.n} rows appended to {LOG_CSV.relative_to(REPO_DIR)} this session | {len(m)} images scored")

## 5. Download results
As in M1, Colab's copy of the repo is deleted when the session ends: download the zip and copy `results/` into your local repo before committing.
Sweep images are saved as JPEG (≈60 KB each) to keep the repo small; the baseline and final set are PNG.

In [ ]:
from google.colab import files
bundle = Path("/content/m2_results")
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(RESULTS_DIR, bundle / "results")
shutil.make_archive("/content/m2_results", "zip", bundle)
print(f"{sum(1 for _ in (bundle / 'results').rglob('*') if _.is_file())} files")
files.download("/content/m2_results.zip")